# 本次复现报告：已经算了什么，还缺什么

日期：2026-10-13。主入口完成后运行本文件。所有表格结果来自前面实际执行的 DataFrame，原论文数字只参与参照，不参与生成计算值。

**当前数据能够完成一套可运行的 RESSET 复现实验，尚不能把论文所有数值与结论标记为精确复现。** 具体限制是历史财务公告不全、扣非及股本字段口径未齐、美国/借壳事件/历史行业分类缺失，以及贝叶斯原图先验校准细节与作者脚本尚不能逐项核对。缺项完整登记，不用近似图表填充原论文图。

## 先看研究主线结论

- CH-3 因子月均收益：市场约0.727%、SMB约0.899%、VMG约0.954%；SMB与VMG月均值t约2.77和3.49。
- 十项正文异象多空组合中，平均绝对alpha由CAPM约0.974%降至CH-3约0.465%，平均R²由约0.061升至约0.470；5%显著alpha由5项降至4项。CH-3提高解释力，但本地结果仍有未解释的异象。
- ROE盈利能力代理因子对CH-3的alpha约0.153%（t=0.93，199月）；资产增长投资代理因子约0.002%（t=0.01，95月）。当前代理样本未拒绝零alpha，不等于证明alpha为零，也不代表精确复现FF-5输入。
- 30%截断依据、估值比率赛马、CH-3/FF-3互相定价和GRS、CH-4换手因子定价属于补充实验，详见主入口后的对应notebook。

## 先看覆盖

“有计算值”不是“与论文一致”；“没有拒绝模型”不是“模型已得到证明”。使用相同键查看原值/计算值/差值，再查看原文204个月与本地实际月数的区别。

In [ ]:
computed_targets = pd.DataFrame(computed_rows)
assert not computed_targets.duplicated(['table','panel','row','metric']).any()
comparison_list = []
for table,target in paper_targets.items():
    reference = target.assign(table=table)
    compared = reference.merge(computed_targets,on=['table','panel','row','metric'],how='left',validate='one_to_one')
    compared['difference'] = compared['computed_value']-compared['paper_value']
    compared['status'] = compared['status'].fillna('缺同口径数据：见扩展覆盖')
    comparison_list.append(compared)
final_comparison = pd.concat(comparison_list,ignore_index=True)
final_coverage = final_comparison[final_comparison['paper_value'].notna()].groupby('table').agg(paper_cells=('paper_value','count'),computed_cells=('computed_value','count'),min_months=('months','min'),max_months=('months','max'))
final_coverage['status'] = np.where(final_coverage['computed_cells'].eq(0),'缺数据，未数值复现',np.where(final_coverage['computed_cells'].lt(final_coverage['paper_cells']),'部分格计算，存在缺项','全部数值格有计算值，仍有口径差异'))
display(final_coverage)
display(extension_coverage)
print('论文原值数值格：',int(final_comparison['paper_value'].notna().sum()))
print('有本地计算的原值格：',int((final_comparison['paper_value'].notna() & final_comparison['computed_value'].notna()).sum()))
print('本地因子实际月份：',f_ch3.shape[0],'；原文目标204')


## 作者公开因子与本地重建分开比较

作者2021更新版月 CH-4 中的市场、价值因子与 CH-3 共用，但规模因子不同，因此只对同名模型的三个/四个因子做来源区分。以下不是重新算出的作者因子，是已经公开的对照输入。更新版与2019论文出版值也可能不同。

作者利率被用作本地重建的基础输入，保留其百分数精度；本地股票排序、组合权重和模型回归仍由本项目数据计算。

In [ ]:
author_column_map = {'mktrf':mkt,'smb':smb,'vmg':vmg,'pmo':'turnover_factor_pmo'}
author_ch4 = author_ch4_monthly.rename(columns=author_column_map)[list(author_column_map.values())]
common = f_ch4.dropna().index.intersection(author_ch4.dropna().index)
author_check_rows = []
for column in f_ch4:
    author_series = author_ch4.loc[common,column]
    local_series = f_ch4.loc[common,column]
    author_check_rows.append({'factor':column,'author_mean_pct':100*author_series.mean(),'local_mean_pct':100*local_series.mean(),'difference_pct':100*(local_series-author_series).mean(),'correlation':local_series.corr(author_series),'months':len(common),'source':'author updated2021 vs local proxy'})
author_validation = pd.DataFrame(author_check_rows)
display(author_validation)
print('作者公开完整204月 MKT/VMG均值（%）：',(100*author_ch4[[mkt,vmg]].mean()).to_dict())


## 剔壳影响与原文未列表结论

0%/25%/30%/35% 剔壳因子已在扩展 notebook 并排计算；这里再检验含壳三因子能否解释剔壳 SMB/VMG。原文目标：含壳大小/价值均值1.36%/0.87%，剔壳因子在含壳模型下alpha约−0.23%/+0.27%；目标是对照项，不能硬编码成结果。

14种原文异象保留在A1；波动率、MAX、换手、ROE、CP、BM、EP及反转的多空方向固定。正文十个异象的显著性同时输出5%/10%阈值。

In [ ]:
if 'shell_cut_sensitivity' in extension_results:
    included_shell = extension_results['shell_cut_sensitivity']['factors'][0.0]
    shell_compare_rows = []
    for column in [smb,vmg]:
        result = run_ts(f_ch3[column],included_shell)
        shell_compare_rows.append({'factor':column,'mean_with_shell_pct':100*included_shell[column].mean(),'alpha_shell_free_on_shell_model_pct':100*result.params['const'],'t_alpha':result.tvalues['const'],'months':int(result.nobs)})
    display(pd.DataFrame(shell_compare_rows))
    shell_test_returns = f_ch3[[smb,vmg]].rename(columns={smb:'test_smb',vmg:'test_vmg'})
    print('剔壳因子对含壳模型的联合GRS：',grs_test(shell_test_returns,included_shell))
display(conclusions)


## 月份缺口：不能悄悄松动15条规则

本地日交易记录显示1999-02、2000-02、2001-01、2002-02、2004-01、2005-02的全市场交易日少于15。严格实施“前一月至少15条”会让相应下一收益月没有任何合格股票。主样本中缺少2000-03、2001-02、2002-03、2004-02、2005-03，得到199个月。

这与论文声称204个月之间是一个需要作者数据/筛选细节澄清的真实差异。本次没有在这些月份放松条件来凑204。滚动解释度按36个自然月重建网格，遇到缺少因子的窗口不压缩时间。

## 不能计算的部分如何恢复

03中的 `extension_inputs` 定义了每一份缺失表的股票代码、主键、日期及金额口径。补入真实数据后主入口自动运行这些检验；缺输入时只登记状态。缺少的核心输入包括：扣非/归母净利润历史报告及公告；1999—2006财务报表；完整A/B/H股本；真实上市日；133起借壳的交易/董事会/审批时间；美国CRSP/Compustat；历史金融及申万37行业分类；2012—2016收盘bid/ask。

Fig A1现在由07的原始解析边际似然生成本地曲线；全候选因子先验校准与原图文字存在范围差异，明确保留限制。不能用GRS或Sharpe排名的softmax替代。

## 全部逐格结果

下面每张表显示所有原值格，不受默认DataFrame截断影响；无计算值格保持NaN。

In [ ]:
display(table9_common_result)
if 'bayesian_comparison' in globals() and 'coverage' in bayesian_comparison:
    display(bayesian_comparison['coverage'])


In [ ]:
for table in ['table{}'.format(i) for i in range(1,11)]+['a{}'.format(i) for i in range(1,6)]:
    print(table)
    display(HTML(final_comparison[final_comparison['table'].eq(table)][['panel','row','metric','paper_value','computed_value','difference','months','status']].to_html(index=False,max_rows=None,max_cols=None)))
